# Chunk Size & Overlap Analysis

**Run this notebook BEFORE `run_ingestion.py`.**

This analysis determines the optimal `CHUNK_SIZE` and `CHUNK_OVERLAP` values to set in `ingestion/neo4j_loader.py`. It works entirely from raw arXiv documents — no Neo4j connection required.

## Why the original value was 32 768

The original `chunk_size=32768` was deliberately chosen to keep each paper as **one or very few chunks**. The intent was whole-document retrieval — send the full paper text as LLM context. This is reasonable for a small corpus but breaks as the collection grows:
- One embedding for 32K chars represents too many topics; cosine similarity loses precision.
- The LLM context window fills fast, leaving room for fewer papers per query.
- Fine-grained claims within a paper cannot be targeted — every query retrieves the whole paper.

## Correct order of analysis

```
1. Sample raw documents from arXiv          (no database needed)
2. Document length distribution             (understand what we are chunking)
3. Structural boundary analysis             (sentence & paragraph distributions)
4. Derive candidate sizes from the data     (anchored to real statistics, not guesses)
5. Chunk count analysis                     (index size & granularity trade-off)
6. Retrieval quality grid                   (faithfulness, precision, latency)
7. Overlap sensitivity                      (does overlap rescue boundary-split answers?)
8. Decision                                 (update neo4j_loader.py with winning config)
```

## 0. Setup

In [1]:
import re
import time
import warnings

import arxiv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        11,
})

EMBED_MODEL   = SentenceTransformer('all-MiniLM-L6-v2')
ARXIV_CLIENT  = arxiv.Client()

# How many papers to sample for the analysis.
# 100 is enough to get a stable distribution; increase for more precision.
SAMPLE_SIZE = 100

print('Setup complete.')

ModuleNotFoundError: No module named 'matplotlib'

## 1. Sample raw documents from arXiv

We fetch a representative sample of `cs.AI` papers — the same category and time range used in the main ingestion pipeline. Full PDF text is loaded so the distribution reflects what will actually be chunked.

This step takes a few minutes depending on network speed. The results are saved to `sample_docs.csv` so you can re-run later sections without re-downloading.

In [ ]:
def fetch_sample_ids(n: int) -> list[str]:
    """Fetch n arXiv IDs from cs.AI (recent papers)."""
    search = arxiv.Search(
        query='cat:cs.AI',
        max_results=n,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    ids = []
    for result in ARXIV_CLIENT.results(search):
        has_pdf = any(link.title == 'pdf' for link in result.links)
        if has_pdf:
            raw = result.entry_id.split('abs/')[-1]
            ids.append(re.sub(r'v\d+$', '', raw))
    return ids


def load_paper(arxiv_id: str) -> dict | None:
    """Load full text + metadata for one paper. Returns None on failure."""
    try:
        loader = ArxivLoader(
            query=arxiv_id,
            load_max_docs=1,
            load_all_available_meta=True,
            load_full_documents=True,
        )
        docs = loader.load()
        if not docs:
            return None
        text = docs[0].page_content
        # Skip papers that are abnormally large (likely mis-classified books)
        if len(text) > 1_000_000:
            return None
        return {
            'entry_id': arxiv_id,
            'title':    docs[0].metadata.get('Title', ''),
            'text':     text,
            'length':   len(text),
        }
    except Exception as exc:
        print(f'  [skip] {arxiv_id}: {exc}')
        return None


print(f'Fetching {SAMPLE_SIZE} paper IDs from arXiv ...')
ids = fetch_sample_ids(SAMPLE_SIZE)
print(f'  Got {len(ids)} IDs. Loading full text ...')

corpus = []
for i, pid in enumerate(ids, 1):
    doc = load_paper(pid)
    if doc:
        corpus.append(doc)
    if i % 10 == 0:
        print(f'  {i}/{len(ids)} loaded, {len(corpus)} successful so far')

print(f'\nLoaded {len(corpus)} papers successfully.')

# Save so downstream cells can re-run without re-fetching
pd.DataFrame([{k: v for k, v in d.items() if k != 'text'} for d in corpus]).to_csv(
    'sample_metadata.csv', index=False
)
print('Metadata saved to sample_metadata.csv')

## 2. Document length distribution

This is the foundation of the entire analysis. We must understand what we are chunking before we can choose any size.

Key questions:
- What is the typical full-text length of a cs.AI paper?
- How wide is the spread? Are there outliers?
- What fraction of papers would fit in a single chunk of size X?

The last question directly explains why 32 768 was chosen originally.

In [ ]:
lengths = np.array([d['length'] for d in corpus])

stats = {
    'count':  int(len(lengths)),
    'min':    int(lengths.min()),
    'p10':    int(np.percentile(lengths, 10)),
    'p25':    int(np.percentile(lengths, 25)),
    'median': int(np.median(lengths)),
    'mean':   int(lengths.mean()),
    'p75':    int(np.percentile(lengths, 75)),
    'p90':    int(np.percentile(lengths, 90)),
    'p95':    int(np.percentile(lengths, 95)),
    'p99':    int(np.percentile(lengths, 99)),
    'max':    int(lengths.max()),
    'std':    int(lengths.std()),
}

print('Document length statistics (characters)\n')
for k, v in stats.items():
    print(f'  {k:<10} {v:>12,}')

print()
print('Single-chunk coverage — what % of papers fit in ONE chunk of this size:')
print()
for cs in [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]:
    pct = 100 * np.mean(lengths <= cs)
    bar = '█' * int(pct / 2)
    print(f'  {cs:>7,}  {pct:5.1f}%  {bar}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
ax = axes[0]
ax.hist(lengths, bins=50, color='#378ADD', alpha=0.8, edgecolor='white', linewidth=0.4)
for val, label, color in [
    (stats['median'], 'median', '#D85A30'),
    (stats['mean'],   'mean',   '#1D9E75'),
    (stats['p90'],    'p90',    '#7F77DD'),
    (stats['p99'],    'p99',    '#BA7517'),
]:
    ax.axvline(val, color=color, linewidth=1.5, linestyle='--', label=f'{label}: {val:,}')
ax.set_xlabel('Document length (chars)')
ax.set_ylabel('Number of papers')
ax.set_title('Full-text length distribution')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=9)

# Box plot
ax2 = axes[1]
ax2.boxplot(
    lengths, vert=False, patch_artist=True,
    boxprops=dict(facecolor='#B5D4F4', color='#185FA5'),
    medianprops=dict(color='#D85A30', linewidth=2),
    whiskerprops=dict(color='#185FA5'),
    capprops=dict(color='#185FA5'),
    flierprops=dict(marker='.', color='#378ADD', alpha=0.4, markersize=3),
)
ax2.set_xlabel('Document length (chars)')
ax2.set_title('Spread & outliers')
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.set_yticks([])

plt.tight_layout()
plt.savefig('1_doc_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Structural boundary analysis — sentences & paragraphs

A chunk should align with natural text boundaries, not split mid-sentence or mid-paragraph. For arXiv papers the hierarchy is:

```
sentence  (~50–250 chars)      smallest coherent unit
paragraph (~200–1 000 chars)   one complete argument
subsection (~1 000–4 000 chars) grouped arguments
section    (~4 000–15 000 chars) Introduction / Methods / Results…
```

**Hard rule:** `chunk_size` must be ≥ p75 of paragraph length, or 25% of paragraphs will be split mid-argument. This is the data-driven floor for our candidates.

In [ ]:
sent_lengths = []
para_lengths = []

for doc in corpus:
    # Sentences: split on terminal punctuation
    for s in re.split(r'(?<=[.!?])\s+', doc['text']):
        s = s.strip()
        if len(s) > 10:
            sent_lengths.append(len(s))

    # Paragraphs: split on blank lines (double newline)
    for p in doc['text'].split('\n\n'):
        p = p.strip()
        if len(p) > 30:
            para_lengths.append(len(p))

sent_arr = np.array(sent_lengths)
para_arr = np.array(para_lengths)

print('Sentence length (chars):')
for pct in [25, 50, 75, 90, 95]:
    print(f'  p{pct:<3} = {np.percentile(sent_arr, pct):>8,.0f}')

print()
print('Paragraph length (chars):')
for pct in [25, 50, 75, 90, 95]:
    print(f'  p{pct:<3} = {np.percentile(para_arr, pct):>8,.0f}')

para_p75 = int(np.percentile(para_arr, 75))
para_med = int(np.median(para_arr))
print()
print(f'► Minimum sensible chunk_size = para p75 = {para_p75:,} chars')
print(f'  Any chunk_size < {para_p75:,} will split 25%+ of paragraphs mid-argument.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, arr, clip, title, xlabel in [
    (axes[0], sent_arr, 1000, 'Sentence length distribution', 'Sentence length (chars)'),
    (axes[1], para_arr, 8000, 'Paragraph length distribution', 'Paragraph length (chars)'),
]:
    ax.hist(arr[arr < clip], bins=50, color='#5DCAA5', alpha=0.8, edgecolor='white', linewidth=0.4)
    for pct, color, label in [
        (50, '#1D9E75', 'p50'), (75, '#D85A30', 'p75'),
        (90, '#7F77DD', 'p90'), (95, '#BA7517', 'p95'),
    ]:
        v = np.percentile(arr, pct)
        ax.axvline(v, color=color, linewidth=1.5, linestyle='--', label=f'{label}: {v:,.0f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('2_structural_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Derive candidate chunk sizes from the distribution

Each candidate is anchored to a real statistic from steps 2–3. No round numbers chosen arbitrarily.

| Candidate | Anchor | Rationale |
|---|---|---|
| `para_p75` | p75 paragraph length | Absolute floor — splits only 25% of paragraphs |
| `para_p90` | p90 paragraph length | Keeps 90% of paragraphs intact |
| `para_p95` | p95 paragraph length | Almost never splits a paragraph |
| `subsection` | 3× median paragraph | Approximates a subsection |
| `section` | 6× median paragraph | Approximates a short section |
| `doc_p25` | p25 document length | 25% of papers fit in one chunk |
| `original` | 32 768 | Original value — reference only, excluded from quality grid |

In [ ]:
def r64(x: float) -> int:
    """Round to nearest 64, minimum 256."""
    return max(256, round(int(x) / 64) * 64)

CANDIDATES = {
    'para_p75':   r64(np.percentile(para_arr, 75)),
    'para_p90':   r64(np.percentile(para_arr, 90)),
    'para_p95':   r64(np.percentile(para_arr, 95)),
    'subsection': r64(3 * para_med),
    'section':    r64(6 * para_med),
    'doc_p25':    r64(np.percentile(lengths, 25)),
    'original':   32768,
}

print(f'  {"candidate":>14}  {"size (chars)":>14}')
print('  ' + '-' * 32)
for name, size in CANDIDATES.items():
    marker = '  ← original (reference only)' if name == 'original' else ''
    print(f'  {name:>14}  {size:>14,}{marker}')

## 5. Chunk count analysis

Before measuring quality, understand what each candidate produces in terms of index size and granularity. Smaller chunks = more nodes in Neo4j = finer-grained retrieval but higher storage and query cost.

In [ ]:
OVERLAP_FRACS = [0.0, 0.10, 0.15]

def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    return RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', ''],
    ).split_text(text)

count_rows = []
for name, size in CANDIDATES.items():
    for frac in OVERLAP_FRACS:
        overlap = int(size * frac)
        counts  = np.array([len(chunk_text(d['text'], size, overlap)) for d in corpus])
        count_rows.append({
            'candidate':    name,
            'size':         size,
            'overlap_pct':  f'{frac*100:.0f}%',
            'overlap':      overlap,
            'chunks_min':   int(counts.min()),
            'chunks_p25':   int(np.percentile(counts, 25)),
            'chunks_median':int(np.median(counts)),
            'chunks_p75':   int(np.percentile(counts, 75)),
            'chunks_max':   int(counts.max()),
            'total_chunks': int(counts.sum()),
        })

cc_df = pd.DataFrame(count_rows)

print('Chunk count summary (0% overlap)\n')
print(cc_df[cc_df['overlap_pct'] == '0%'][
    ['candidate', 'size', 'chunks_median', 'chunks_p75', 'chunks_max', 'total_chunks']
].to_string(index=False))

In [ ]:
base = cc_df[cc_df['overlap_pct'] == '0%'].copy()
colors = ['#F0997B' if c == 'original' else '#B5D4F4' for c in base['candidate']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

bars1 = axes[0].bar(base['candidate'], base['chunks_median'], color=colors, edgecolor='white', linewidth=0.5)
axes[0].bar_label(bars1, fmt='%d', fontsize=9, padding=3)
axes[0].set_ylabel('Median chunks per paper')
axes[0].set_title('Median chunk count per paper (0% overlap)')
axes[0].set_xticklabels(base['candidate'], rotation=20, ha='right')

bars2 = axes[1].bar(base['candidate'], base['total_chunks'], color=colors, edgecolor='white', linewidth=0.5)
axes[1].bar_label(bars2, labels=[f'{v:,}' for v in base['total_chunks']], fontsize=8, padding=3)
axes[1].set_ylabel('Total Chunk nodes in index')
axes[1].set_title(f'Total Neo4j Chunk nodes across {len(corpus)} papers (0% overlap)')
axes[1].set_xticklabels(base['candidate'], rotation=20, ha='right')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('3_chunk_counts.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Retrieval quality grid search

We exclude `original` (32 768) from the quality grid — at that size most papers produce 1 chunk, making it whole-document retrieval. Comparing it against fine-grained candidates is apples vs oranges.

### QA probe construction
We extract sentence-level `(query, gold_span)` pairs from the raw paper text. Each sentence between 60–400 chars becomes one probe. The gold span is a verbatim substring of the source, so ground truth is exact. We use 100 probes.

### Metrics
| Metric | Definition |
|---|---|
| **Faithfulness** | Is the gold span present in (or semantically equivalent to) the top-3 retrieved chunks? |
| **Context precision** | What fraction of retrieved characters belong to the gold span? |
| **p95 latency** | 95th-percentile wall-clock retrieval time (embed + cosine rank), ms |

In [ ]:
def build_qa_probe(docs: list[dict], max_pairs: int = 100) -> list[dict]:
    pairs = []
    for doc in docs:
        for s in re.split(r'(?<=[.!?])\s+', doc['text']):
            s = s.strip()
            if 60 < len(s) < 400 and len(pairs) < max_pairs:
                pairs.append({
                    'query':       f'What does the paper say about: "{s[:80]}"',
                    'gold_span':   s,
                    'source_text': doc['text'],
                })
    return pairs


def retrieve_top_k(query: str, chunks: list[str], k: int = 3) -> list[str]:
    if not chunks:
        return []
    scores = cosine_similarity(
        EMBED_MODEL.encode([query]),
        EMBED_MODEL.encode(chunks)
    )[0]
    return [chunks[i] for i in np.argsort(scores)[::-1][:k]]


def faithfulness(retrieved: list[str], gold: str, threshold: float = 0.82) -> float:
    for c in retrieved:
        if gold.lower() in c.lower():
            return 1.0
        sim = cosine_similarity(
            EMBED_MODEL.encode([gold]),
            EMBED_MODEL.encode([c])
        )[0][0]
        if sim >= threshold:
            return 1.0
    return 0.0


def context_precision(retrieved: list[str], gold: str) -> float:
    total = sum(len(c) for c in retrieved)
    if not total:
        return 0.0
    overlap = sum(len(gold) for c in retrieved if gold.lower() in c.lower())
    return min(overlap / total, 1.0)


qa_probe = build_qa_probe(corpus, max_pairs=100)
print(f'Built {len(qa_probe)} QA probe pairs from raw document text.')

In [ ]:
quality_rows = []
eval_candidates = {k: v for k, v in CANDIDATES.items() if k != 'original'}
total = len(eval_candidates) * len(OVERLAP_FRACS)
done  = 0

for name, size in eval_candidates.items():
    for frac in OVERLAP_FRACS:
        overlap = int(size * frac)
        f_sc, p_sc, lats = [], [], []

        for probe in qa_probe:
            t0   = time.perf_counter()
            topk = retrieve_top_k(
                probe['query'],
                chunk_text(probe['source_text'], size, overlap),
                k=3,
            )
            lats.append((time.perf_counter() - t0) * 1000)
            f_sc.append(faithfulness(topk, probe['gold_span']))
            p_sc.append(context_precision(topk, probe['gold_span']))

        quality_rows.append({
            'candidate':         name,
            'size':              size,
            'overlap':           overlap,
            'overlap_pct':       f'{frac*100:.0f}%',
            'faithfulness':      round(float(np.mean(f_sc)),  3),
            'context_precision': round(float(np.mean(p_sc)),  3),
            'latency_p50_ms':    round(float(np.percentile(lats, 50)), 1),
            'latency_p95_ms':    round(float(np.percentile(lats, 95)), 1),
        })
        done += 1
        r = quality_rows[-1]
        print(f'[{done:2}/{total}] {name:>14}  size={size:>6,}  overlap={overlap:>5,}  '
              f'faith={r["faithfulness"]:.3f}  prec={r["context_precision"]:.3f}  '
              f'p95={r["latency_p95_ms"]:5.1f}ms')

quality_df = pd.DataFrame(quality_rows)
print('\nGrid search complete.')

## 7. Full results table with composite score

In [ ]:
# Composite = 60% faithfulness + 30% context precision + 10% inverted-normalised latency
max_lat = quality_df['latency_p95_ms'].max()
quality_df['composite'] = (
    0.60 * quality_df['faithfulness'] +
    0.30 * quality_df['context_precision'] +
    0.10 * (1 - quality_df['latency_p95_ms'] / max_lat)
).round(3)

best_idx = quality_df['composite'].idxmax()

def highlight(row):
    if row.name == best_idx:
        return ['background-color:#d4edda; font-weight:bold'] * len(row)
    return [''] * len(row)

quality_df.sort_values('composite', ascending=False).style.apply(highlight, axis=1)

## 8. Six-panel quality dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
ax_f, ax_p, ax_l, ax_sc, ax_fp, ax_ov = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(3)]

palette = {'0%': '#378ADD', '10%': '#1D9E75', '15%': '#D85A30'}
names   = list(eval_candidates.keys())

for frac_str, color in palette.items():
    sub = quality_df[quality_df['overlap_pct'] == frac_str].sort_values('size')
    ax_f.plot(sub['size'], sub['faithfulness'],       marker='o', color=color, label=frac_str, linewidth=1.5)
    ax_p.plot(sub['size'], sub['context_precision'],  marker='o', color=color, label=frac_str, linewidth=1.5)
    ax_l.plot(sub['size'], sub['latency_p95_ms'],     marker='o', color=color, label=frac_str, linewidth=1.5)
    ax_fp.scatter(sub['faithfulness'], sub['context_precision'], color=color, label=frac_str, s=55)
    sub2 = quality_df[quality_df['overlap_pct'] == frac_str].sort_values('size')
    ax_sc.plot(sub2['size'], sub2['composite'], marker='o', color=color, label=frac_str, linewidth=1.5)

best_row = quality_df.loc[best_idx]
ax_sc.scatter([best_row['size']], [best_row['composite']], color='#D85A30', s=140, zorder=5,
              label=f'best ({best_row["candidate"]})')

x = np.arange(len(names))
for i, (frac_str, color) in enumerate(palette.items()):
    sub = quality_df[quality_df['overlap_pct'] == frac_str].set_index('candidate')
    vals = [float(sub.loc[n, 'faithfulness']) if n in sub.index else 0 for n in names]
    ax_ov.bar(x + i * 0.25, vals, 0.25, label=frac_str, color=color, alpha=0.85)
ax_ov.set_xticks(x + 0.25)
ax_ov.set_xticklabels(names, rotation=22, ha='right', fontsize=8)

for ax, title, xlabel, ylabel in [
    (ax_f,  'Faithfulness vs chunk size',        'size (chars)', 'faithfulness'),
    (ax_p,  'Context precision vs chunk size',    'size (chars)', 'context precision'),
    (ax_l,  'p95 latency vs chunk size',          'size (chars)', 'latency (ms)'),
    (ax_sc, 'Composite score vs chunk size',      'size (chars)', 'composite'),
    (ax_fp, 'Faithfulness vs precision trade-off','faithfulness', 'context precision'),
    (ax_ov, 'Faithfulness by overlap fraction',   'candidate',    'faithfulness'),
]:
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8)

fig.suptitle('Chunk size & overlap quality analysis — all-MiniLM-L6-v2', fontsize=13, y=1.01)
plt.savefig('4_chunk_quality_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Overlap sensitivity — does it actually rescue boundary-split answers?

Overlap has one specific job: when a relevant sentence straddles a chunk boundary, the 0%-overlap split loses it. Overlap duplicates the boundary region so both adjacent chunks contain the crossing text.

We measure how often 15% overlap rescues an answer that 0% overlap misses, on the best-scoring chunk size.

In [ ]:
best_size    = int(best_row['size'])
best_overlap = int(best_size * 0.15)
rescued      = 0

for probe in qa_probe:
    top_0  = retrieve_top_k(probe['query'], chunk_text(probe['source_text'], best_size, 0),            k=3)
    top_15 = retrieve_top_k(probe['query'], chunk_text(probe['source_text'], best_size, best_overlap), k=3)
    if faithfulness(top_0, probe['gold_span']) == 0.0 and faithfulness(top_15, probe['gold_span']) == 1.0:
        rescued += 1

pct = 100 * rescued / len(qa_probe)

base_total    = int(cc_df[(cc_df['candidate'] == best_row['candidate']) & (cc_df['overlap_pct'] == '0%' )]['total_chunks'].values[0])
overlap_total = int(cc_df[(cc_df['candidate'] == best_row['candidate']) & (cc_df['overlap_pct'] == '15%')]['total_chunks'].values[0])

print(f'Best candidate size          : {best_size:,} chars')
print(f'15% overlap value            : {best_overlap:,} chars')
print(f'Answers rescued by overlap   : {rescued}/{len(qa_probe)} ({pct:.1f}%)')
print()
print(f'Index size at  0% overlap    : {base_total:,} Chunk nodes')
print(f'Index size at 15% overlap    : {overlap_total:,} Chunk nodes  (+{overlap_total - base_total:,})')
print()
print('Decision rule:')
print('  rescued% > 2%  →  overlap is worth the extra storage.')
print('  rescued% < 1%  →  0% overlap is sufficient for this chunk size.')

## 10. Decision & instructions to update ingestion

In [ ]:
best = quality_df.loc[best_idx]

print('=' * 62)
print('  RECOMMENDED CONFIGURATION')
print('=' * 62)
print(f'  chunk_size        = {best["size"]:,}')
print(f'  chunk_overlap     = {best["overlap"]:,}  ({best["overlap_pct"]} of chunk_size)')
print(f'  candidate name    = {best["candidate"]}')
print(f'  faithfulness      = {best["faithfulness"]}')
print(f'  context_precision = {best["context_precision"]}')
print(f'  latency p95       = {best["latency_p95_ms"]} ms')
print(f'  composite score   = {best["composite"]}')
print('=' * 62)
print()
print('Steps to apply:')
print('  1. Open ingestion/neo4j_loader.py')
print(f'     Set CHUNK_SIZE    = {best["size"]}')
print(f'     Set CHUNK_OVERLAP = {best["overlap"]}')
print('  2. Run the ingestion pipeline:')
print('     python run_ingestion.py')
print()
print('Why the original 32 768 was sub-optimal for chunk retrieval:')
pct_single = 100 * np.mean(lengths <= 32768)
print(f'  {pct_single:.1f}% of sampled papers fit in one chunk at size 32 768.')
print('  One embedding for an entire paper cannot precisely match a')
print('  specific claim — it averages across every topic in the paper.')

# Export all results for records
quality_df.to_csv('chunk_quality_results.csv', index=False)
cc_df.to_csv('chunk_count_results.csv',        index=False)
pd.DataFrame([stats]).to_csv('doc_length_stats.csv', index=False)
print('\nAll results exported to CSV files.')